In [5]:
"""
Calculate accuracy for one or more classification_results_*.xlsx files
against your manually-entered ground truth column, the same comparison
done manually earlier in this project.

Handles:
  - Different ground truth column names across files (correct_label,
    correct label, label.1, etc.) -- auto-detects the first match.
  - Minor label spelling variants the models sometimes produce
    (e.g. "relevante", "relevance") by matching on the "relevan"/"irrelevan"
    prefix rather than requiring an exact string match.
  - parse_failed / API failed rows -- reported separately, not silently
    counted as wrong.
  - Prints a clean summary table across all files, and a per-model
    breakdown of which papers were wrong and their trap_reason.

Usage: edit FILES below (or leave as-is if your filenames match), then run.
Optionally saves a combined summary to accuracy_summary.xlsx.
"""

import pandas as pd

FILES = {
    #"deepseek": "classification_results_deep-seek.xlsx",
    "gemini": "LLM-FULL/classification_results_gemini-flash.xlsx",
    "gpt": "LLM-FULL/classification_results_terra.xlsx",
    #"llama": "classification_results_llama.xlsx",
    "mistral": "LLM-FULL/classification_results_mistral.xlsx",
    "qwen": "LLM-FULL/classification_results_qwen.xlsx",
    "sonnet": "LLM-FULL/classification_results_sonnet.xlsx",
}

TRUTH_COLUMN_CANDIDATES = ["correct_label", "correct label", "label.1", "ground_truth", "truth"]


def normalize_label(val) -> str:
    """Collapse label spelling variants (relevant/relevante/relevance) to a
    consistent 'relevant'/'irrelevant'/'' for comparison purposes."""
    s = str(val).strip().lower()
    if s.startswith("irrelevan"):
        return "irrelevant"
    if s.startswith("relevan"):
        return "relevant"
    return s


def find_truth_column(columns) -> str | None:
    for candidate in TRUTH_COLUMN_CANDIDATES:
        normalized_candidate = candidate.strip().lower().replace(" ", "_")
        for col in columns:
            if col.strip().lower().replace(" ", "_") == normalized_candidate:
                return col
    return None


def score_file(name: str, path: str):
    try:
        df = pd.read_excel(path)
    except FileNotFoundError:
        print(f"{name}: file not found ({path}), skipping.")
        return None, None

    truth_col = find_truth_column(df.columns)
    n_total = len(df)

    n_parse_failed = df["parse_failed"].fillna(False).astype(bool).sum() if "parse_failed" in df.columns else 0
    n_no_label = df["label"].isna().sum() if "label" in df.columns else n_total

    if truth_col is None:
        print(f"{name}: no ground truth column found (checked {TRUTH_COLUMN_CANDIDATES}), "
              f"cannot compute accuracy.")
        return {"model": name, "n_total": n_total, "n_scored": None, "accuracy": None,
                "n_parse_failed": n_parse_failed, "n_wrong": None}, None

    df["_label_norm"] = df["label"].apply(normalize_label)
    df["_truth_norm"] = df[truth_col].apply(normalize_label)

    # Only score rows that actually have both a predicted label and a truth
    # label -- exclude parse-failed/empty rows from the accuracy denominator
    # rather than silently counting them as wrong.
    scoreable = df[(df["_label_norm"] != "") & (df["_label_norm"] != "nan")
                   & (df["_truth_norm"] != "") & (df["_truth_norm"] != "nan")]

    n_scored = len(scoreable)
    n_correct = (scoreable["_label_norm"] == scoreable["_truth_norm"]).sum()
    accuracy = n_correct / n_scored if n_scored > 0 else None

    wrong = scoreable[scoreable["_label_norm"] != scoreable["_truth_norm"]]
    n_wrong = len(wrong)

    print(f"\n=== {name} ===")
    print(f"Total rows: {n_total} | Scored: {n_scored} | Correct: {n_correct} | "
          f"Wrong: {n_wrong} | Parse failed/unscoreable: {n_total - n_scored}")
    print(f"Accuracy (on scoreable rows): {accuracy:.1%}" if accuracy is not None else "Accuracy: N/A")

    if n_wrong > 0:
        cols_to_show = [c for c in ["wos_id", "title", "label", truth_col, "trap_reason"] if c in wrong.columns]
        print(f"\nWrong rows for {name}: (see accuracy_details.xlsx, sheet '{name}_wrong', for full list)")
        print(wrong[cols_to_show].head(5).to_string(index=False))
        if n_wrong > 5:
            print(f"  ... and {n_wrong - 5} more (full list saved to file)")

    wrong_export = wrong[cols_to_show].copy() if n_wrong > 0 else pd.DataFrame(
        columns=[c for c in ["wos_id", "title", "label", truth_col, "trap_reason"] if c in df.columns]
    )

    return {
        "model": name,
        "n_total": n_total,
        "n_scored": n_scored,
        "n_correct": n_correct,
        "n_wrong": n_wrong,
        "accuracy": accuracy,
        "n_parse_failed": n_parse_failed,
    }, wrong_export


def main():
    results = []
    wrong_rows_by_model = {}

    for name, path in FILES.items():
        summary, wrong = score_file(name, path)
        if summary is not None:
            results.append(summary)
        if wrong is not None and len(wrong) > 0:
            wrong_rows_by_model[name] = wrong

    summary_df = pd.DataFrame(results)
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(summary_df.to_string(index=False))

    # Write everything into one workbook: a summary sheet, plus one sheet
    # per model listing every single wrong row (not just the first few
    # printed to console), so nothing is lost to terminal scrollback.
    with pd.ExcelWriter("accuracy_details.xlsx", engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="summary", index=False)
        for name, wrong_df in wrong_rows_by_model.items():
            # Excel sheet names are capped at 31 characters
            sheet_name = f"{name}_wrong"[:31]
            wrong_df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"\nSaved summary + full wrong-row detail (one sheet per model) "
          f"to accuracy_details.xlsx")


if __name__ == "__main__":
    main()


=== gemini ===
Total rows: 433 | Scored: 433 | Correct: 413 | Wrong: 20 | Parse failed/unscoreable: 0
Accuracy (on scoreable rows): 95.4%

Wrong rows for gemini: (see accuracy_details.xlsx, sheet 'gemini_wrong', for full list)
             wos_id                                                                                                                                                                                                                                     title      label correct_label          trap_reason
WOS:000961010200002                                                                                                           Effect of soy protein isolate concentration and whipping time on physicochemical and functional properties of strawberry powder irrelevant      relevant wrong_property_focus
WOS:001384172100001                                                                                                                   Ultrasound-Assisted Enzymatic Extracti

In [9]:
"""
Calculate accuracy for one or more classification_results_*.xlsx files
against your manually-entered ground truth column, the same comparison
done manually earlier in this project.

Handles:
  - Different ground truth column names across files (correct_label,
    correct label, label.1, etc.) -- auto-detects the first match.
  - Minor label spelling variants the models sometimes produce
    (e.g. "relevante", "relevance") by matching on the "relevan"/"irrelevan"
    prefix rather than requiring an exact string match.
  - parse_failed / API failed rows -- reported separately, not silently
    counted as wrong.
  - Prints a clean summary table across all files, and a per-model
    breakdown of which papers were wrong and their trap_reason.

Usage: edit FILES below (or leave as-is if your filenames match), then run.
Optionally saves a combined summary to accuracy_summary.xlsx.
"""

import pandas as pd

FILES = {
    #"deepseek": "classification_results_deep-seek.xlsx",
    "gemini": "LLM-FULL\classification_results_gemini-flash.xlsx",
    "gpt": "LLM-FULL\classification_results_terra.xlsx",
    #"llama": "classification_results_llama.xlsx",
    "mistral": "LLM-FULL\classification_results_mistral.xlsx",
    "qwen": "LLM-FULL\classification_results_qwen.xlsx",
    "sonnet": "LLM-FULL\classification_results_sonnet.xlsx",
}

TRUTH_COLUMN_CANDIDATES = ["correct_label", "correct label", "label.1", "ground_truth", "truth"]


def normalize_label(val) -> str:
    """Collapse label spelling variants (relevant/relevante/relevance) to a
    consistent 'relevant'/'irrelevant'/'' for comparison purposes."""
    s = str(val).strip().lower()
    if s.startswith("irrelevan"):
        return "irrelevant"
    if s.startswith("relevan"):
        return "relevant"
    return s


def find_truth_column(columns) -> str | None:
    for candidate in TRUTH_COLUMN_CANDIDATES:
        normalized_candidate = candidate.strip().lower().replace(" ", "_")
        for col in columns:
            if col.strip().lower().replace(" ", "_") == normalized_candidate:
                return col
    return None


def score_file(name: str, path: str):
    try:
        df = pd.read_excel(path)
    except FileNotFoundError:
        print(f"{name}: file not found ({path}), skipping.")
        return None, None

    truth_col = find_truth_column(df.columns)
    n_total = len(df)

    n_parse_failed = df["parse_failed"].fillna(False).astype(bool).sum() if "parse_failed" in df.columns else 0
    n_no_label = df["label"].isna().sum() if "label" in df.columns else n_total

    if truth_col is None:
        print(f"{name}: no ground truth column found (checked {TRUTH_COLUMN_CANDIDATES}), "
              f"cannot compute accuracy.")
        return {"model": name, "n_total": n_total, "n_scored": None, "accuracy": None,
                "n_parse_failed": n_parse_failed, "n_wrong": None}, None

    df["_label_norm"] = df["label"].apply(normalize_label)
    df["_truth_norm"] = df[truth_col].apply(normalize_label)

    # Only score rows that actually have both a predicted label and a truth
    # label -- exclude parse-failed/empty rows from the accuracy denominator
    # rather than silently counting them as wrong.
    scoreable = df[(df["_label_norm"] != "") & (df["_label_norm"] != "nan")
                   & (df["_truth_norm"] != "") & (df["_truth_norm"] != "nan")]

    n_scored = len(scoreable)
    n_correct = (scoreable["_label_norm"] == scoreable["_truth_norm"]).sum()
    accuracy = n_correct / n_scored if n_scored > 0 else None

    wrong = scoreable[scoreable["_label_norm"] != scoreable["_truth_norm"]]
    n_wrong = len(wrong)

    print(f"\n=== {name} ===")
    print(f"Total rows: {n_total} | Scored: {n_scored} | Correct: {n_correct} | "
          f"Wrong: {n_wrong} | Parse failed/unscoreable: {n_total - n_scored}")
    print(f"Accuracy (on scoreable rows): {accuracy:.1%}" if accuracy is not None else "Accuracy: N/A")

    if n_wrong > 0:
        cols_to_show = [c for c in ["wos_id", "title", "label", truth_col, "trap_reason"] if c in wrong.columns]
        print(f"\nWrong rows for {name}: (see accuracy_details.xlsx, sheet '{name}_wrong', for full list)")
        print(wrong[cols_to_show].head(5).to_string(index=False))
        if n_wrong > 5:
            print(f"  ... and {n_wrong - 5} more (full list saved to file)")

    wrong_export = wrong[cols_to_show].copy() if n_wrong > 0 else pd.DataFrame(
        columns=[c for c in ["wos_id", "title", "label", truth_col, "trap_reason"] if c in df.columns]
    )

    return {
        "model": name,
        "n_total": n_total,
        "n_scored": n_scored,
        "n_correct": n_correct,
        "n_wrong": n_wrong,
        "accuracy": accuracy,
        "n_parse_failed": n_parse_failed,
    }, wrong_export


def main():
    results = []
    wrong_rows_by_model = {}

    for name, path in FILES.items():
        summary, wrong = score_file(name, path)
        if summary is not None:
            results.append(summary)
        if wrong is not None and len(wrong) > 0:
            wrong_rows_by_model[name] = wrong

    summary_df = pd.DataFrame(results)
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(summary_df.to_string(index=False))

    # Write everything into one workbook: a summary sheet, plus one sheet
    # per model listing every single wrong row (not just the first few
    # printed to console), so nothing is lost to terminal scrollback.
    with pd.ExcelWriter("LLM-FULL\\accuracy_details.xlsx", engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="summary", index=False)
        for name, wrong_df in wrong_rows_by_model.items():
            # Excel sheet names are capped at 31 characters
            sheet_name = f"{name}_wrong"[:31]
            wrong_df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"\nSaved summary + full wrong-row detail (one sheet per model) "
          f"to accuracy_details.xlsx")


if __name__ == "__main__":
    main()

<>:25: SyntaxWarning: invalid escape sequence '\c'
<>:26: SyntaxWarning: invalid escape sequence '\c'
<>:28: SyntaxWarning: invalid escape sequence '\c'
<>:29: SyntaxWarning: invalid escape sequence '\c'
<>:30: SyntaxWarning: invalid escape sequence '\c'
<>:25: SyntaxWarning: invalid escape sequence '\c'
<>:26: SyntaxWarning: invalid escape sequence '\c'
<>:28: SyntaxWarning: invalid escape sequence '\c'
<>:29: SyntaxWarning: invalid escape sequence '\c'
<>:30: SyntaxWarning: invalid escape sequence '\c'
C:\Users\olagunju\AppData\Local\Temp\ipykernel_40808\1734272669.py:25: SyntaxWarning: invalid escape sequence '\c'
  "gemini": "LLM-FULL\classification_results_gemini-flash.xlsx",
C:\Users\olagunju\AppData\Local\Temp\ipykernel_40808\1734272669.py:26: SyntaxWarning: invalid escape sequence '\c'
  "gpt": "LLM-FULL\classification_results_terra.xlsx",
C:\Users\olagunju\AppData\Local\Temp\ipykernel_40808\1734272669.py:28: SyntaxWarning: invalid escape sequence '\c'
  "mistral": "LLM-FULL\cl


=== gemini ===
Total rows: 433 | Scored: 433 | Correct: 419 | Wrong: 14 | Parse failed/unscoreable: 0
Accuracy (on scoreable rows): 96.8%

Wrong rows for gemini: (see accuracy_details.xlsx, sheet 'gemini_wrong', for full list)
             wos_id                                                                                                                                                                                                                                     title      label correct_label                    trap_reason
WOS:000961010200002                                                                                                           Effect of soy protein isolate concentration and whipping time on physicochemical and functional properties of strawberry powder irrelevant      relevant           wrong_property_focus
WOS:000394511400023 Effect of different treatments on the microstructure and functional and pasting properties of pigeon pea (Cajanus cajan L.), dolicho